# Train and evaluate a calibrated ModernBERT LLM router

ModernBERT predicts **whether a faster candidate preserves the quality of a strong fallback**. A deterministic analytical model—not ModernBERT and not candidate inference—estimates latency from model facts and prompt size.

The default run is a **random-split feasibility test**: can prompt content predict safe replacement at all? Repeated prompt content is grouped by a normalized SHA-256 hash, so the same question cannot cross train, validation, and test under different benchmark IDs. After random feasibility succeeds across several seeds, rerun with `SPLIT_MODE = "dataset_ood"` as a separate generalization stress test.

A successful sealed-test POC must activate the router, retain at least 98% quality at a one-sided 95% lower confidence bound, produce positive net analytical latency savings after router overhead, and route at least some prompts away from fallback. Validation uses an additional 1 percentage-point safety margin: it must reach a 99% LCB before the 98% sealed-test gate is opened.

> This proves feasibility under explicit analytical assumptions. It does not claim measured production latency.

## 1. One-cell Google Colab setup

Open this notebook in Google Colab, select a GPU runtime, and run every cell in order. This cell clones the `develop` branch, installs the project normally (not as an editable package), registers `src` in the live kernel, and verifies the import immediately. No terminal, runtime restart, or separate setup notebook is required.

In [1]:
%cd /content
!test -d /content/LLM_Router || git clone --branch develop https://github.com/BrunoVitti96/LLM-router.git /content/LLM_Router
!git -C /content/LLM_Router pull --ff-only origin develop
%cd /content/LLM_Router
%pip install -q -U ".[notebook]"

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/LLM_Router")
SOURCE_ROOT = str(PROJECT_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
for module_name in list(sys.modules):
    if module_name == "llm_router" or module_name.startswith("llm_router."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import llm_router

print(f"Router package ready from {Path(llm_router.__file__).resolve()}")

/content
Cloning into '/content/LLM_Router'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 222 (delta 110), reused 173 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 4.76 MiB | 6.24 MiB/s, done.
Resolving deltas: 100% (110/110), done.
From https://github.com/BrunoVitti96/LLM-router
 * branch            develop    -> FETCH_HEAD
Already up to date.
/content/LLM_Router
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Experiment controls and automatic benchmark download

Start with `random`. Five stratified content-group folds produce an approximate 60/20/20 split while keeping repeated prompts together. Use `dataset_ood` only for a separate, harder artifact. For a serious conclusion, repeat both modes with at least seeds 42, 43, and 44. Never tune a later run from an already-opened test result.

The official archive is about 1.28 GB. The loader discovers its `dataset/split/model/file.json` structure instead of assuming a fixed wrapper name. No candidate LLM is installed or executed.

In [2]:
import shutil
import tarfile
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display

from llm_router.config import DEFAULT_CONFIG
from llm_router.modernbert_poc import (
    export_modernbert_hybrid_poc,
    train_modernbert_hybrid_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    simulate_economics,
    split_benchmark,
)
from llm_router.utils.training import seed_everything

SEED = 42
SPLIT_MODE = "random"  # Use "dataset_ood" only for a separate stress test.
EPOCHS = 8  # Maximum; validation early stopping usually finishes sooner.
MINIMUM_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 2
VALIDATION_QUALITY_MARGIN = DEFAULT_CONFIG.validation_quality_margin
MINIMUM_MACRO_QUALITY_RETENTION = (
    DEFAULT_CONFIG.minimum_macro_quality_retention
)
MAXIMUM_QUALITY_LOSS_RATE_UCL = DEFAULT_CONFIG.maximum_quality_loss_rate_ucl
MINIMUM_ROUTED_SAFETY_PRECISION_LCB = (
    DEFAULT_CONFIG.minimum_routed_safety_precision_lcb
)
MINIMUM_GUARDED_DATASET_QUALITY_LCB = (
    DEFAULT_CONFIG.minimum_guarded_dataset_quality_retention_lcb
)
MINIMUM_GUARDED_DATASET_PROMPTS = (
    DEFAULT_CONFIG.minimum_guarded_dataset_prompts
)
CONSERVATIVE_ROUTER_OVERHEAD_S = (
    DEFAULT_CONFIG.conservative_router_overhead_s
)
ROUTER_CONFIG = replace(DEFAULT_CONFIG, seed=SEED)
DATA_ROOT = Path("/content/LLMRouterBench")
OUTPUT_DIR = Path(
    f"reports_benchmark/modernbert_hybrid_{SPLIT_MODE}_seed_{SEED}"
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
inventory = benchmark_inventory(DATA_ROOT)
archive_preview = []
if inventory.empty:
    archive = hf_hub_download(
        repo_id="NPULH/LLMRouterBench",
        filename="bench-release.tar.gz",
        repo_type="dataset",
    )
    results_dir = DATA_ROOT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as bundle:
        archive_preview = bundle.getnames()[:12]
        bundle.extractall(results_dir, filter="data")
    inventory = benchmark_inventory(DATA_ROOT)
if inventory.empty:
    extracted_preview = [
        str(path.relative_to(DATA_ROOT))
        for path in DATA_ROOT.rglob("*")
        if path.is_file()
    ][:20]
    raise RuntimeError(
        "No dataset/split/model/*.json benchmark layout was found after "
        f"extraction. Archive entries: {archive_preview}. "
        f"Extracted files: {extracted_preview}"
    )
DATA_ROOT = Path(inventory.iloc[0]["file"]).parents[3]
print({
        "device": DEVICE,
        "split_mode": SPLIT_MODE,
        "seed": SEED,
        "data_root": str(DATA_ROOT.resolve()),
        "output_dir": str(OUTPUT_DIR),
    })
if DEVICE == "cpu":
    print("Warning: CPU training works for a smoke test but will be slow. A GPU is recommended.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


bench-release.tar.gz:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

{'device': 'cuda', 'split_mode': 'random', 'seed': 42, 'data_root': '/content/LLMRouterBench/results/bench-release', 'output_dir': 'reports_benchmark/modernbert_hybrid_random_seed_42'}


## 3. Inspect available datasets and models

Do this before editing model profiles. The strings in `SELECTED_MODELS` must exactly match model directory names shown below. Choose at least two candidates and at least three datasets for a dataset-disjoint split.

In [3]:
inventory = benchmark_inventory(DATA_ROOT)
assert not inventory.empty, "No LLMRouterBench result files were found."
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
display(inventory_summary)
print(f"Available models: {inventory.model.nunique()}")
print(f"Available datasets: {inventory.dataset.nunique()}")

,model,dataset,source_split,files
0,DeepHermes-3-Llama-3-8B-Preview,aime,hybrid,1
1,DeepHermes-3-Llama-3-8B-Preview,arcc,test,1
2,DeepHermes-3-Llama-3-8B-Preview,bbh,test,1
3,DeepHermes-3-Llama-3-8B-Preview,emorynlp,test,1
4,DeepHermes-3-Llama-3-8B-Preview,finqa,test,1
...,...,...,...,...
562,qwen3-235b-a22b-thinking-2507,mmlupro,test_1000,1
563,qwen3-235b-a22b-thinking-2507,mmlupro,test_3000,1
564,qwen3-235b-a22b-thinking-2507,simpleqa,subset_500,1
565,qwen3-235b-a22b-thinking-2507,simpleqa,test,1


Available models: 40
Available datasets: 24


## 4. Declare a non-dominated candidate panel

The previous run included NVIDIA-Nemotron-Nano-9B-v2, but it was analytically slower than the Qwen fallback and had a 0% oracle-selection rate. It could never change the latency-minimizing decision.

The clean default is therefore:

- `Fin-R1`: approximately 7B parameters, the faster replacement candidate;
- `Qwen3-8B`: 8.2B parameters, the potential quality fallback.

Both generate autoregressively at BF16 precision. The public pool contains no diffusion model; add one only when matching pre-collected quality results and sourced denoising settings exist.

In [4]:
MODEL_PROFILES = {
    "Fin-R1": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "Qwen3-8B": {
        "parameters_billions": 8.2,
        "active_parameters_billions": 8.2,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
}
SELECTED_MODELS = tuple(MODEL_PROFILES)
assert len(SELECTED_MODELS) >= 2, "Routing requires at least two models."
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, (
    f"Configured models were not found: {missing_models}. "
    f"Available models: {sorted(inventory.model.unique())}"
)
print(SELECTED_MODELS)

('Fin-R1', 'Qwen3-8B')


## 5. Load pre-collected quality and calculate analytical latency

The benchmark provides candidate answers and quality scores. `simulate_economics` calculates latency without loading those candidates. Expected output length is a bounded function of prompt length, so the policy cannot peek at a candidate's realized response length. The panel also records a normalized prompt hash and duplicate-group size for content-level leakage auditing.

In [5]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=pd.Timestamp.utcnow().date().isoformat(),
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()
print({"complete_prompts": len(panel.examples), "models": panel.models})
print({
    "repeated_prompt_rows": int((panel.examples.prompt_group_size > 1).sum()),
    "largest_prompt_group": int(panel.examples.prompt_group_size.max()),
})
display(
    simulated.groupby("model")["simulated_latency_s"]
    .agg(["min", "median", "mean", "max"])
    .sort_values("mean")
)

{'complete_prompts': 14041, 'models': ('Fin-R1', 'Qwen3-8B')}
{'repeated_prompt_rows': 135, 'largest_prompt_group': 4}


,min,median,mean,max
model,,,,
Fin-R1,0.662578,1.277956,1.707854,4.870356
Qwen3-8B,0.691318,1.412189,1.920396,5.696962


### Leakage check

This deliberately changes every realized completion length. Analytical latency must remain identical because only prompt size and declared model/scenario properties are allowed to affect it.

In [6]:
counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
), "Analytical latency unexpectedly depends on realized completion length."
print("Passed: candidate completion tokens do not affect analytical latency.")

Passed: candidate completion tokens do not affect analytical latency.


## 6. Freeze train, validation, and sealed-test splits

The fallback is selected from training quality only. Validation is used for checkpoint selection, out-of-fold calibration, threshold selection, and the activation gate. Test outcomes remain sealed until the complete policy is frozen.

- `random`: normalized-prompt-hash-disjoint feasibility test; start here.
- `dataset_ood`: dataset-disjoint generalization stress test; run separately.

In [7]:
split = split_benchmark(panel, mode=SPLIT_MODE, seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            split.train_datasets,
            split.validation_datasets,
            split.test_datasets,
        ],
    }
)
display(split_summary)
assert not set(split.train).intersection(split.validation)
assert not set(split.train).intersection(split.test)
assert not set(split.validation).intersection(split.test)
split_hashes = {
    name: set(panel.examples.iloc[indices].prompt_hash)
    for name, indices in {
        "train": split.train,
        "validation": split.validation,
        "test": split.test,
    }.items()
}
assert split_hashes["train"].isdisjoint(split_hashes["validation"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["validation"].isdisjoint(split_hashes["test"])
if SPLIT_MODE == "dataset_ood":
    assert set(split.train_datasets).isdisjoint(split.validation_datasets)
    assert set(split.train_datasets).isdisjoint(split.test_datasets)
    assert set(split.validation_datasets).isdisjoint(split.test_datasets)

,split,prompts,datasets
0,train,8425,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."
1,validation,2804,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."
2,test,2812,"(aime, arcc, bbh, emorynlp, finqa, gpqa, human..."


## 7. Verify validation-only oracle headroom and scenario sensitivity

The hindsight oracle sees recorded outcomes and chooses the fastest model that matches or beats fallback quality. It is unattainable at deployment, but it proves whether routing opportunity exists.

This cell uses **validation only** and repeats the calculation under conservative compute, conservative bandwidth, longer-output, and larger fixed-overhead assumptions. If modest changes erase headroom, the latency claim is too fragile to train. Router-overhead sensitivity is evaluated later on the frozen test policy because it depends on how often the router selects the faster model.

In [8]:
def validation_oracle_metrics(candidate_panel):
    fallback_index = int(candidate_panel.score[split.train].mean(axis=0).argmax())
    target = oracle_choices(
        candidate_panel.score,
        candidate_panel.latency,
        fallback_index=fallback_index,
    )
    indices = split.validation
    rows = np.arange(len(indices))
    chosen = target[indices]
    fallback_quality = candidate_panel.score[indices, fallback_index].mean()
    oracle_quality = candidate_panel.score[indices][rows, chosen].mean()
    fallback_latency = candidate_panel.latency[indices, fallback_index].mean()
    oracle_latency = candidate_panel.latency[indices][rows, chosen].mean()
    return {
        "fallback_model": candidate_panel.models[fallback_index],
        "quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "latency_savings": 1 - oracle_latency / fallback_latency,
        "fallback_usage": np.mean(chosen == fallback_index),
    }

sensitivity_settings = {
    "balanced": {},
    "compute_conservative": {"effective_tflops": 40.0},
    "bandwidth_conservative": {"memory_bandwidth_gbps": 600.0},
    "larger_fixed_overhead": {"fixed_model_overhead_s": 0.050},
    "longer_outputs": {
        "output_base_tokens": 48.0,
        "output_tokens_per_prompt_token": 0.35,
    },
}
sensitivity_rows = []
for name, overrides in sensitivity_settings.items():
    variant = replace(scenario, name=name, **overrides)
    variant_records = simulate_economics(records, variant)
    variant_panel = make_complete_panel(variant_records, models=SELECTED_MODELS)
    assert variant_panel.examples.example_id.equals(panel.examples.example_id)
    sensitivity_rows.append(
        {"scenario": name, **validation_oracle_metrics(variant_panel)}
    )
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
assert (sensitivity.latency_savings > 0).all(), (
    "Oracle headroom is not robust across the declared sensitivity scenarios."
)

,scenario,fallback_model,quality_retention,latency_savings,fallback_usage
0,balanced,Qwen3-8B,1.091599,0.084319,0.242154
1,compute_conservative,Qwen3-8B,1.091599,0.084130,0.242154
2,bandwidth_conservative,Qwen3-8B,1.091599,0.084671,0.242154
3,larger_fixed_overhead,Qwen3-8B,1.091599,0.082834,0.242154
4,longer_outputs,Qwen3-8B,1.091599,0.083940,0.242154


## 8. Understand the objective, loss, and calibration

For each non-fallback candidate, replacement safety is:

$$y_m(x)=\mathbf 1[Q_m(x)\ge Q_f(x)-\epsilon_q].$$

The deployable head now uses **class-balanced** binary cross-entropy. Candidate $m$ receives positive weight $N_{unsafe}/N_{safe}$, clipped for numerical stability. This prevents a high safe-rate prior from winning without learning prompt differences.

A second, training-only head imitates the hindsight oracle and penalizes expected quality risk plus latency regret:

$$\mathcal L_{oracle}=(1+g_o)CE(z,o)+4\sum_m p_m d_m+\sum_m p_m r_m.$$

The complete objective is:

$$\boxed{\mathcal L_{train}=\mathcal L_{balanced\ safety}+0.25\mathcal L_{oracle}}.$$

After checkpoint selection, each safety logit receives per-candidate Platt calibration. Threshold search uses **out-of-fold validation probabilities**, so a validation row never calibrates itself. The exported deployment scaler is fitted on all validation rows. ModernBERT never predicts latency or the final model.

## 9. Train ModernBERT

Exact dataset names are excluded from router inputs. ModernBERT sees prompt text and prompt-token count, reducing brittle dataset-identity memorization. The LoRA adapter uses a lower learning rate than the newly initialized heads, and validation early stopping prevents the falling training loss from hiding overfitting. On FP16 GPUs, skipped optimizer steps do not advance the learning-rate scheduler.

The input diagnostics use ModernBERT's own tokenizer. For example, if `truncation_rate=0.12`, then 12% of prompts exceeded the 512-token router limit; that would motivate a longer limit or chunked encoding in a later experiment.

In [9]:
training = train_modernbert_hybrid_poc(
    panel,
    split,
    config=ROUTER_CONFIG,
    epochs=EPOCHS,
    batch_size=8,
    learning_rate=1e-4,
    head_learning_rate=2e-4,
    minimum_epochs=MINIMUM_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    quality_epsilon=0.0,
    safety_loss_weight=1.0,
    oracle_auxiliary_weight=0.25,
    device=DEVICE,
)
display(training.history)
display(training.calibration_diagnostics)
display(pd.DataFrame([training.input_diagnostics]))
print({
    "best_epoch": training.best_epoch,
    "epochs_completed": training.epochs_completed,
    "stopped_early": training.stopped_early,
    "training_seconds": round(training.training_seconds, 1),
})
assert training.safety_probabilities.shape == panel.score.shape
assert np.allclose(
    training.safety_probabilities[:, training.fallback_index], 1.0
)
assert np.all(
    (training.safety_probabilities >= 0)
    & (training.safety_probabilities <= 1)
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/596M [00:00<?, ?B/s]

W0819 13:22:16.568000 830 torch/_inductor/utils.py:1731] [1/0_1] Not enough SMs to use max_autotune_gemm mode


,epoch,train_total_loss,train_safety_loss,train_oracle_auxiliary_loss,validation_total_loss,validation_safety_loss,validation_oracle_auxiliary_loss,skipped_optimizer_steps,encoder_learning_rate,head_learning_rate
0,1,0.572115,0.298465,1.094602,0.587416,0.308233,1.116731,2,0.000098,0.000197
1,2,0.528561,0.276835,1.006906,0.568669,0.296596,1.088293,2,0.000089,0.000179
2,3,0.510176,0.267655,0.970086,0.562030,0.294218,1.071247,3,0.000074,0.000148
3,4,0.491704,0.258351,0.933412,0.581435,0.308787,1.090591,3,0.000054,0.000108
4,5,0.466278,0.245739,0.882156,0.579038,0.305422,1.094466,4,0.000034,0.000068


,candidate,validation_examples,safe_prevalence,constant_brier,raw_brier,calibrated_brier,raw_brier_skill,calibrated_brier_skill,raw_ece,calibrated_ece,safe_roc_auc,safe_average_precision,unsafe_average_precision
0,Fin-R1,2804,0.757846,0.183515,0.178748,0.150962,0.02598,0.177385,0.133495,0.023488,0.739123,0.881457,0.520742


,examples,max_input_tokens,truncated_examples,truncation_rate,router_tokens_p50,router_tokens_p95,router_tokens_max
0,14041,512,474,0.033758,85.0,463.0,1408


{'best_epoch': 3, 'epochs_completed': 5, 'stopped_early': True, 'training_seconds': 2056.4}


## 10. Freeze the policy, then open the sealed test

Validation selects the confidence threshold from a dense, predeclared grid and may disable the router. Selection requires `98% + 1% margin = 99%` aggregate validation LCB, at least 98% macro-dataset LCB, at most a 2.5% quality-loss-rate UCL, at least a 90% routed-safety-precision LCB, at least a 90% worst-dataset LCB among datasets with 100 or more prompts, and positive analytical savings at both 4 ms and a conservative 20 ms router overhead. Only after that policy is frozen does the report evaluate test outcomes against the corresponding sealed-test gates.

Numerical example: 92% routed safety precision can look adequate, but if its one-sided 95% lower bound is only 89%, the policy remains disabled. Likewise, 2.3% savings at 4 ms do not qualify if they turn negative at 20 ms.

In [10]:
result = run_public_benchmark(
    panel,
    split,
    objective="latency",
    minimum_quality_retention=0.98,
    confidence=0.95,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    minimum_predicted_savings=0.02,
    router_overhead_s=scenario.router_overhead_s,
    conservative_router_overhead_s=CONSERVATIVE_ROUTER_OVERHEAD_S,
    minimum_macro_quality_retention=MINIMUM_MACRO_QUALITY_RETENTION,
    maximum_quality_loss_rate_ucl=MAXIMUM_QUALITY_LOSS_RATE_UCL,
    minimum_routed_safety_precision_lcb=(
        MINIMUM_ROUTED_SAFETY_PRECISION_LCB
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        MINIMUM_GUARDED_DATASET_QUALITY_LCB
    ),
    minimum_guarded_dataset_prompts=MINIMUM_GUARDED_DATASET_PROMPTS,
    seed=SEED,
    routing_probabilities=training.safety_probabilities,
    router_name="modernbert_hybrid_router",
    decision_metadata={
        "router_input_tokens": training.router_input_lengths,
        "router_was_truncated": training.router_was_truncated,
    },
)
assert result.fallback_model == panel.models[training.fallback_index]
display(result.threshold_search)
display(result.summary)
display(result.candidate_diagnostics)
router_dataset_metrics = result.per_dataset_metrics.loc[
    result.per_dataset_metrics.strategy.eq("modernbert_hybrid_router")
].sort_values("quality_retention")
display(router_dataset_metrics)
overhead_sensitivity = result.router_overhead_sensitivity
display(overhead_sensitivity)
print({
    "break_even_router_overhead_ms": round(
        overhead_sensitivity.break_even_router_overhead_ms.iloc[0], 2
    )
})
print(
    {
        "poc_passed": result.poc_passed,
        "router_active": result.router_active,
        "selected_threshold": result.selected_threshold,
        "fallback_model": result.fallback_model,
        "failure_reasons": result.failure_reasons,
    }
)

,threshold,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,routed_safety_precision_lcb,safe_opportunity_recall,...,macro_dataset_quality_retention_lcb,macro_dataset_count,guarded_dataset_quality_retention,guarded_dataset_quality_retention_lcb,guarded_dataset_count,mean_resource,fallback_resource,resource_savings,conservative_resource_savings,fallback_usage
0,0.500,0.608060,0.704708,0.862854,0.842515,0.159415,0.171114,0.817253,0.804052,0.940706,...,0.771285,18.0,0.680556,0.548783,12.0,1.770106,1.952785,0.093548,0.085354,0.127675
1,0.600,0.631598,0.704708,0.896255,0.876980,0.134807,0.145769,0.836151,0.823083,0.907765,...,0.843928,18.0,0.750000,0.618466,12.0,1.783178,1.952785,0.086854,0.078660,0.177247
2,0.700,0.651926,0.704708,0.925101,0.907125,0.110913,0.121045,0.852676,0.839537,0.847059,...,0.905199,18.0,0.813830,0.618466,12.0,1.798714,1.952785,0.078898,0.070705,0.247147
3,0.750,0.663338,0.704708,0.941296,0.924413,0.094864,0.104361,0.860221,0.846634,0.770353,...,0.926027,18.0,0.833333,0.618466,12.0,1.816558,1.952785,0.069760,0.061567,0.321327
4,0.800,0.673324,0.704708,0.955466,0.939511,0.081669,0.090585,0.862048,0.847536,0.673412,...,0.942337,18.0,0.847222,0.599816,12.0,1.831342,1.952785,0.062190,0.053996,0.407989
5,0.850,0.685449,0.704708,0.972672,0.959171,0.056705,0.064325,0.868486,0.851673,0.494118,...,0.956968,18.0,0.864516,0.714482,12.0,1.862660,1.952785,0.046152,0.037959,0.568830
6,0.855,0.689729,0.704708,0.978745,0.965810,0.050642,0.057897,0.875110,0.858087,0.468235,...,0.962106,18.0,0.877419,0.733027,12.0,1.867082,1.952785,0.043888,0.035694,0.594508
7,0.860,0.690799,0.704708,0.980263,0.967734,0.047432,0.054482,0.875117,0.857498,0.438588,...,0.961809,18.0,0.888889,0.751832,12.0,1.872029,1.952785,0.041355,0.033161,0.620185
8,0.865,0.693295,0.704708,0.983806,0.972042,0.041369,0.048010,0.880041,0.861786,0.400471,...,0.962345,18.0,0.895833,0.775403,12.0,1.878973,1.952785,0.037798,0.029605,0.655136
9,0.870,0.697575,0.704708,0.989879,0.978838,0.034950,0.041118,0.889888,0.871433,0.372706,...,0.969925,18.0,0.923611,0.827919,12.0,1.884475,1.952785,0.034981,0.026788,0.682596


,objective,selected_model,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,routed_safety_precision_lcb,...,macro_dataset_count,guarded_dataset_quality_retention,guarded_dataset_quality_retention_lcb,guarded_dataset_count,mean_resource,fallback_resource,resource_savings,conservative_resource_savings,fallback_usage,oracle_savings_capture
best_single,latency,Qwen3-8B,0.713371,0.713371,1.000000,1.000000,0.000000,0.000961,1.000000,1.000000,...,18.0,1.000000,1.000000,13.0,1.884562,1.884562,0.000000,0.000000,1.000000,0.000000
cheapest_single,latency,Fin-R1,0.548720,0.713371,0.769192,0.746394,0.233286,0.246657,0.766714,0.753343,...,18.0,0.085526,-0.045951,13.0,1.677601,1.884562,0.109819,0.109819,0.000000,1.337521
dataset_lookup,latency,dynamic,0.725462,0.713371,1.016949,1.009633,0.008179,0.011484,0.900862,0.863824,...,18.0,1.000000,1.000000,13.0,1.877295,1.884562,0.003856,0.003856,0.917496,0.046960
outcome_oracle,latency,dynamic,0.782006,0.713371,1.096211,1.085216,0.000000,0.000961,1.000000,0.998747,...,18.0,1.010101,0.983179,13.0,1.729827,1.884562,0.082107,0.082107,0.233286,1.000000
modernbert_hybrid_router,latency,dynamic,0.715861,0.713371,1.003490,0.998239,0.006046,0.008970,0.946032,0.921039,...,18.0,0.975410,0.944416,13.0,1.864732,1.884562,0.010522,0.002032,0.887980,0.128153


,split,model,is_fallback,mean_quality,mean_resource,relative_resource_vs_fallback,safe_rate,candidate_better_rate,candidate_equal_rate,candidate_worse_rate,both_zero_rate,faster_than_fallback_rate,oracle_selection_rate
0,train,Fin-R1,False,0.541484,1.708869,0.889306,0.768427,0.062315,0.706113,0.231573,0.226944,1.0,0.768427
1,train,Qwen3-8B,True,0.710742,1.921576,1.000000,1.000000,0.000000,1.000000,0.000000,0.289258,0.0,0.231573
2,validation,Fin-R1,False,0.527104,1.735142,0.888547,0.757846,0.064551,0.693295,0.242154,0.230742,1.0,0.757846
3,validation,Qwen3-8B,True,0.704708,1.952785,1.000000,1.000000,0.000000,1.000000,0.000000,0.295292,0.0,0.242154
4,test,Fin-R1,False,0.548720,1.677601,0.890181,0.766714,0.068634,0.698080,0.233286,0.217994,1.0,0.766714
5,test,Qwen3-8B,True,0.713371,1.884562,1.000000,1.000000,0.000000,1.000000,0.000000,0.286629,0.0,0.233286


,strategy,dataset,prompts,quality,fallback_quality,quality_retention,quality_retention_lcb,quality_loss_rate,quality_loss_rate_ucl,routed_safety_precision,...,macro_dataset_quality_retention_lcb,macro_dataset_count,guarded_dataset_quality_retention,guarded_dataset_quality_retention_lcb,guarded_dataset_count,mean_resource,fallback_resource,resource_savings,conservative_resource_savings,fallback_usage
73,modernbert_hybrid_router,arcc,259,0.918919,0.942085,0.975410,0.956473,0.027027,0.049115,0.951389,...,0.975410,1.0,0.975410,0.956473,1.0,0.873785,0.900231,0.029376,0.011603,0.444015
76,modernbert_hybrid_router,finqa,223,0.708520,0.721973,0.981366,0.958563,0.017937,0.039354,0.913043,...,0.981366,1.0,0.981366,0.958563,1.0,4.220092,4.338614,0.027318,0.023630,0.793722
75,modernbert_hybrid_router,emorynlp,152,0.296053,0.296053,1.000000,1.000000,0.000000,0.017488,1.000000,...,1.000000,1.0,1.000000,1.000000,1.0,1.847711,1.843711,-0.002170,-0.010848,1.000000
72,modernbert_hybrid_router,aime,11,0.909091,0.909091,1.000000,1.000000,0.000000,0.197405,1.000000,...,1.000000,1.0,1.000000,1.000000,0.0,1.293669,1.289669,-0.003102,-0.015508,1.000000
78,modernbert_hybrid_router,humaneval,31,0.580645,0.580645,1.000000,1.000000,0.000000,0.080270,1.000000,...,1.000000,1.0,1.000000,1.000000,0.0,1.750678,1.755690,0.002854,-0.006259,0.935484
77,modernbert_hybrid_router,gpqa,39,0.769231,0.769231,1.000000,1.000000,0.000000,0.064873,1.000000,...,1.000000,1.0,1.000000,1.000000,0.0,1.333514,1.329514,-0.003009,-0.015043,1.000000
79,modernbert_hybrid_router,kandk,138,0.768116,0.768116,1.000000,1.000000,0.000000,0.019228,1.000000,...,1.000000,1.0,1.000000,1.000000,1.0,1.340871,1.336871,-0.002992,-0.014960,1.000000
80,modernbert_hybrid_router,korbench,246,0.569106,0.569106,1.000000,0.983351,0.004065,0.018013,0.973684,...,1.000000,1.0,1.000000,0.983351,1.0,3.477443,3.570230,0.025989,0.021508,0.845528
84,modernbert_hybrid_router,mathbench,29,0.965517,0.965517,1.000000,1.000000,0.000000,0.085333,1.000000,...,1.000000,1.0,1.000000,1.000000,0.0,1.058861,1.054861,-0.003792,-0.018960,1.000000
81,modernbert_hybrid_router,livecodebench,222,0.684685,0.684685,1.000000,1.000000,0.000000,0.012040,1.000000,...,1.000000,1.0,1.000000,1.000000,1.0,2.656569,2.652569,-0.001508,-0.007540,1.000000


,router_overhead_ms,net_latency_savings,break_even_router_overhead_ms
0,4.0,0.010522,23.829731
1,10.0,0.007338,23.829731
2,20.0,0.002032,23.829731
3,50.0,-0.013887,23.829731


{'break_even_router_overhead_ms': np.float64(23.83)}
{'poc_passed': True, 'router_active': True, 'selected_threshold': 0.91, 'fallback_model': 'Qwen3-8B', 'failure_reasons': ()}


### Interpret the result honestly

- **Oracle savings near zero:** the panel or assumptions offer no useful headroom.
- **Oracle headroom but `router_active=False`:** ModernBERT found no validation-safe policy.
- **Router active but `poc_passed=False`:** validation looked promising, but sealed-test behavior did not generalize.
- **Random passes and dataset-OOD fails:** feasibility exists, but cross-domain generalization does not.
- **Both modes pass across several seeds and sensitivity scenarios:** this is a credible analytical-latency POC.

A fallback-only outcome proves the safety guard worked; it does not prove the router worked.

## 11. Export the complete report and router artifact

The export contains strategy metrics, threshold search, sealed-test decisions, full per-candidate safety probabilities, candidate and per-dataset diagnostics, harm-rate and routed-precision confidence bounds, safe-opportunity recall, analytical and router-overhead sensitivity, per-prompt input-truncation diagnostics, explicit pass/fail reasons, training history, calibration and ranking diagnostics, Platt parameters, LoRA weights, both heads, and tokenizer.

Numerical example: a 2% pooled saving can still be unacceptable if one small dataset retains only 90% quality, or if a measured 50 ms router overhead exceeds the reported break-even overhead. The additional CSVs make both problems visible.

In [11]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
sensitivity.to_csv(report_dir / "validation_sensitivity.csv", index=False)
artifact_dir = export_modernbert_hybrid_poc(
    training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    poc_passed=result.poc_passed,
    failure_reasons=result.failure_reasons,
    minimum_predicted_savings=0.02,
    validation_quality_margin=VALIDATION_QUALITY_MARGIN,
    minimum_macro_quality_retention=MINIMUM_MACRO_QUALITY_RETENTION,
    maximum_quality_loss_rate_ucl=MAXIMUM_QUALITY_LOSS_RATE_UCL,
    minimum_routed_safety_precision_lcb=(
        MINIMUM_ROUTED_SAFETY_PRECISION_LCB
    ),
    minimum_guarded_dataset_quality_retention_lcb=(
        MINIMUM_GUARDED_DATASET_QUALITY_LCB
    ),
    minimum_guarded_dataset_prompts=MINIMUM_GUARDED_DATASET_PROMPTS,
    conservative_router_overhead_s=CONSERVATIVE_ROUTER_OVERHEAD_S,
    config=ROUTER_CONFIG,
)
print({"reports": str(report_dir.resolve()), "artifact": str(artifact_dir.resolve())})

{'reports': '/content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42', 'artifact': '/content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42/modernbert_router'}


## 12. Download the result before Colab shuts down

The ZIP path contains split mode and seed, so feasibility and dataset-OOD artifacts cannot silently overwrite one another.

In [12]:
bundle_path = shutil.make_archive(
    str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR
)
print(f"Created {bundle_path}")
try:
    from google.colab import files

    files.download(bundle_path)
except ImportError:
    print("Not running in Colab; download the ZIP from the printed path.")

Created /content/LLM_Router/reports_benchmark/modernbert_hybrid_random_seed_42.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Final POC checklist

Before presenting a result, confirm:

- random feasibility passed across at least three seeds;
- dataset-OOD was run as a separate, clearly labeled stress-test artifact;
- every retained alternative has a non-zero oracle-selection rate;
- normalized prompt hashes are disjoint across random train, validation, and test;
- calibration improved or did not materially worsen Brier score, Brier skill, ECE, unsafe average precision, and AUROC;
- macro retention, guarded-dataset retention, harm-rate UCL, routed-safety-precision LCB, and safe-opportunity recall are acceptable;
- ModernBERT input truncation is reported and investigated if material;
- no candidate inference or realized completion length was used for latency;
- fallback selection used training quality only;
- checkpointing, calibration, threshold selection, and activation used validation only;
- test outcomes were opened only after policy freeze;
- net savings include router overhead and remain positive across plausible overhead values;
- conclusions survive all declared sensitivity scenarios; and
- every latency claim says **analytical** or **simulated**, never measured.